# END Engine — graded pilot tasks: END vs DIRECT

Runs the four v0.3.5.1 pilot tasks, which have automatic checkers, under DIRECT and FULL_END, using the Engine
fixes on branch `codex/colab-run5-engine-fixes`. Each answer is scored with the pilot's own evaluator, and the
results are printed next to the historical pilot (unfixed engine, same model build, same tasks and seeds).

| Task | Type | Scored by |
| --- | --- | --- |
| BA-001 | Celsius to Fahrenheit | known answer |
| BA-002 | fix a buggy function | executable tests |
| BA-008 | misleading premise | executable tests + premise check |
| BA-009 | incident with missing evidence | evidence validator |

**How to use:** Runtime → Change runtime type → **L4** (a T4 or A100 also works) → Save → Connect, then
Runtime → Run all. Expect roughly 30–90 minutes depending on the GPU. Keep the tab open and the PC awake.

This is an exploratory run, one trial per task. It is not a preregistered Study A/B result.

**Local fixes applied on top of the branch (not on GitHub yet):** negated sentences such as "we don't have the
metrics to confirm" and purpose clauses such as "to confirm the cause" are no longer treated as overclaims (one
withheld a correct BA-009 answer), and a draft that breaks the task's output format now gets an Engine-derived
revision instruction naming the violation (structured BA-009 never restored its JSON block), and a draft that
would be withheld for overclaims can no longer replace a clean best draft (BA-009 native returned an empty answer
although an earlier draft was correct). Set `APPLY_LOCAL_FIXES = False` to run the branch unchanged.

## 1. Settings

In [ ]:
# --- Edit these if you want -------------------------------------------------
REPO_URL = "https://github.com/kaamikazi/The-End-Engine-.git"
BRANCH = "codex/colab-run5-engine-fixes"   # the four fixes
MODEL = "qwen3:4b"            # the pilot model; compare like with like
MAX_CYCLES = 3                # pilot value for FULL_END
NUM_CTX = 16384               # pilot used 8192; 16384 avoids cut prompts
RUN_OFFLINE_TESTS = True      # the repo's own unit tests, no model needed
SAVE_TO_DRIVE = False         # True = copy the results zip to Google Drive (asks for permission)
DRIVE_FOLDER = "END_Engine_Colab_Runs"
ALLOW_CPU = False             # True = continue without a GPU (very slow)
APPLY_LOCAL_FIXES = True      # False = run the GitHub code unchanged (for comparison)
PATCH_B64 = "..."  # fixes made against commit 6338ccb8dd; do not edit

GRADED_TASKS = ("BA-001", "BA-002", "BA-008", "BA-009")
CONDITIONS = ("DIRECT", "FULL_END_NATIVE", "FULL_END_STRUCTURED")   # drop one to save time
PILOT_SEED = 20261005          # the pilot's seed; per-task seeds match its runs
PILOT_MODEL_DIGEST = "359d7dd4bcdab3d86b87d73ac27966f4dbb9f5efdfcc75d34a8764a09474fae7"
TASK = "graded pilot tasks"
# -----------------------------------------------------------------------------

import datetime, os, pathlib

WORK = pathlib.Path("/content")
REPO = WORK / "end_engine_repo"
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = WORK / "end_runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY = {"run_id": RUN_ID, "model": MODEL, "branch": BRANCH, "max_cycles": MAX_CYCLES, "num_ctx": NUM_CTX, "task": TASK}
print("Results folder:", RUN_DIR)

## 2. GPU check

In [ ]:
import shutil, subprocess

def sh(cmd, check=True, **kw):
    """Run a shell command, print its output and return it."""
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    out = (p.stdout or "") + (p.stderr or "")
    if out.strip():
        print(out.rstrip())
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}")
    return out

if shutil.which("nvidia-smi"):
    gpu = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", check=False).strip()
    sh("nvidia-smi > " + str(RUN_DIR / "nvidia_smi_start.txt"), check=False)
else:
    gpu = ""
SUMMARY["gpu"] = gpu or None
if gpu:
    print("GPU found:", gpu)
elif ALLOW_CPU:
    print("No GPU. Continuing on CPU because ALLOW_CPU = True. This will be slow.")
else:
    raise SystemExit("No GPU attached. Use Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all again. "
                     "(Or set ALLOW_CPU = True in section 1.)")

## 3. Get the END Engine code

In [ ]:
import json, sys

if not (REPO / ".git").exists():
    sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}")
else:
    print("Repo already cloned; reusing it.")
COMMIT = sh(f"git -C {REPO} rev-parse HEAD").strip()
SUMMARY["commit"] = COMMIT

# --- Apply the local Engine fixes (not on GitHub yet) before anything imports end_engine -----------------
import base64, hashlib, zlib
PATCH = json.loads(zlib.decompress(base64.b64decode(PATCH_B64)))
if APPLY_LOCAL_FIXES:
    def file_hash(path):
        return hashlib.sha256((REPO / path).read_bytes()).hexdigest() if (REPO / path).exists() else None
    pending = [p for p, i in PATCH["files"].items() if file_hash(p) != i["sha256"]]
    if pending and "end_engine" in sys.modules:
        raise SystemExit("end_engine was already imported unpatched in this session. "
                         "Use Runtime -> Restart session, then Run all.")
    for path in pending:
        item, target, current = PATCH["files"][path], REPO / path, file_hash(path)
        if current != item["base_sha256"]:
            raise SystemExit(f"{path} in the clone is not the version these fixes were made against "
                             f"(base commit {PATCH['base_commit'][:10]}). Not applying anything.")
        target.write_bytes(base64.b64decode(item["content"]))
        print("patched", path)
    SUMMARY["local_fixes"] = {"base_commit": PATCH["base_commit"],
                              "files": {p: i["sha256"] for p, i in PATCH["files"].items()}}
else:
    SUMMARY["local_fixes"] = None
    print("Local fixes NOT applied (APPLY_LOCAL_FIXES = False).")

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Python", sys.version.split()[0])
if sys.version_info < (3, 11):
    raise SystemExit("END Engine needs Python 3.11 or newer.")

import end_engine
print("END Engine imported from", pathlib.Path(end_engine.__file__).parent)

## 4. Offline unit tests

These use the engine's mock model, so they test the code, not the GPU. A few may fail on Linux if they depend on Windows file behaviour; the run continues either way.

In [ ]:
if RUN_OFFLINE_TESTS:
    p = subprocess.run([sys.executable, "-B", "-m", "unittest", "discover", "-s", "tests", "-t", "."],
                       cwd=REPO, text=True, capture_output=True)
    (RUN_DIR / "offline_tests.txt").write_text(p.stdout + p.stderr)
    tail = (p.stdout + p.stderr).strip().splitlines()[-4:]
    print("\n".join(tail))
    SUMMARY["offline_tests"] = {"passed": p.returncode == 0, "tail": tail}
    print("\nOffline tests:", "PASS" if p.returncode == 0 else "SOME FAILURES (see offline_tests.txt in the results)")
else:
    SUMMARY["offline_tests"] = "skipped"

## 5. Install Ollama and load the model on the GPU

In [ ]:
import json, time, urllib.request

def ollama_up():
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2) as r:
            return json.loads(r.read())["version"]
    except Exception:
        return None

if not shutil.which("ollama"):
    sh("apt-get -qq update && apt-get -qq install -y pciutils zstd > /dev/null", check=False)
    sh("curl -fsSL https://ollama.com/install.sh | sh > /content/ollama_install.log 2>&1")
    print("Ollama installed.")

if not ollama_up():
    OLLAMA_LOG = open(RUN_DIR / "ollama_serve.log", "a")
    subprocess.Popen(["ollama", "serve"], stdout=OLLAMA_LOG, stderr=subprocess.STDOUT,
                     env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
OLLAMA_VERSION = ollama_up()
if not OLLAMA_VERSION:
    raise SystemExit("Ollama server did not start. Check ollama_serve.log in the results folder.")
print("Ollama server version:", OLLAMA_VERSION)
SUMMARY["ollama_version"] = OLLAMA_VERSION

print(f"Pulling {MODEL} (first time takes a minute or two)...")
sh(f"ollama pull {MODEL} 2>&1 | tail -n 2")

# Warm-up call: loads the model into VRAM so we can confirm where it runs.
req = urllib.request.Request("http://127.0.0.1:11434/api/generate", headers={"Content-Type": "application/json"},
    data=json.dumps({"model": MODEL, "prompt": "Say OK.", "stream": False, "think": False,
                     "options": {"num_ctx": NUM_CTX, "num_predict": 8}}).encode())
with urllib.request.urlopen(req, timeout=600) as r:
    warm = json.loads(r.read())
print("Warm-up reply:", repr(warm.get("response", "")[:40]))
ps = sh("ollama ps", check=False)
SUMMARY["ollama_ps"] = ps.strip()
if gpu and "GPU" not in ps:
    print("WARNING: Ollama does not report the model on the GPU. The run will work but be slow. See ollama_serve.log.")

## 6. Run the graded tasks

Each run prints its score as it finishes. FULL_END runs print progress every 30 s.

In [ ]:
import json, subprocess, threading, time, traceback, csv
from v0351_core_regression_pilot.bench.providers import CallLog, ModelSettings, OllamaModel, discover
from v0351_core_regression_pilot.bench.conditions import EndSettings, run_direct, run_full_end
from v0351_core_regression_pilot.bench.upstream_runner import evaluate_run, trial_seed
from v0351_core_regression_pilot.bench.evaluators import evaluate

class GpuSampler:
    """Polls nvidia-smi every 2 s during a run (peak memory, mean utilisation); prints progress every 30 s."""
    def __init__(self, log):
        self.log, self.samples, self._stop = log, [], threading.Event()
    def _loop(self):
        t0, ticks = time.time(), 0
        while not self._stop.is_set():
            if gpu:
                try:
                    line = subprocess.run("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader,nounits",
                                          shell=True, text=True, capture_output=True, timeout=10).stdout.strip().splitlines()[0]
                    self.samples.append(tuple(int(x.strip()) for x in line.split(",")))
                except Exception:
                    pass
            ticks += 1
            if ticks % 15 == 0:
                print(f"  ... {int(time.time() - t0)} s, {len(self.log.entries)} model calls so far", flush=True)
            self._stop.wait(2)
    def __enter__(self):
        threading.Thread(target=self._loop, daemon=True).start()
        return self
    def __exit__(self, *exc):
        self._stop.set()
    def report(self):
        if not self.samples:
            return None
        return {"peak_memory_mib": max(m for m, _ in self.samples),
                "mean_utilization_pct": round(sum(u for _, u in self.samples) / len(self.samples), 1)}

PILOT = REPO / "v0351_core_regression_pilot"
settings = ModelSettings(model=MODEL, num_ctx=NUM_CTX)
identity = discover(settings)
SUMMARY["model_identity"] = identity
SUMMARY["same_model_build_as_pilot"] = identity.get("digest") == PILOT_MODEL_DIGEST
end_settings = EndSettings(full_max_cycles=MAX_CYCLES)
for sub in ("trajectories", "trials", "evaluations"):
    (RUN_DIR / sub).mkdir(exist_ok=True)
HISTORICAL = {r["run_id"]: r for r in csv.DictReader(open(PILOT / "results.csv", encoding="utf-8"))}
RESULTS = []

for task_id in GRADED_TASKS:
    task = json.loads((PILOT / "tasks" / f"{task_id}.json").read_text(encoding="utf-8"))
    for condition in CONDITIONS:
        run_id = f"{task_id}.T1.{condition}"
        log = CallLog()
        model = OllamaModel(settings, identity, log, trial_seed=trial_seed(PILOT_SEED, task_id, 1))
        print(f"\n=== {run_id} ===", flush=True)
        start, error = time.time(), None
        with GpuSampler(log) as sampler:
            try:
                output = (run_direct(task, model) if condition == "DIRECT" else
                          run_full_end(task, model, RUN_DIR / "trajectories", end_settings,
                                       "native" if condition == "FULL_END_NATIVE" else "structured"))
            except Exception:
                output, error = {"answer": "", "stages": {}}, traceback.format_exc()
                print("RUN FAILED:", error.strip().splitlines()[-1])
        end = output.get("end")
        record = {"run_id": run_id, "task_id": task_id, "condition": condition, "answer": output["answer"],
                  "stages": output["stages"], "end": end, "error": error, "calls": log.entries,
                  "cost": {**log.totals(), "wall_ms": round((time.time() - start) * 1000, 3),
                           "end_cycles": (end or {}).get("iterations"), "branch_count": (end or {}).get("branch_count"),
                           "tool_check_calls": 0},
                  "gpu": sampler.report(), "model": model.metadata()}
        (RUN_DIR / "trials" / f"{run_id}.json").write_text(json.dumps(record, indent=1, ensure_ascii=False, default=str))
        try:  # same scoring as the pilot (run_pilot.evaluate_record)
            if end and (not end.get("candidates") or end.get("returned_candidate_id") is None):
                evaluation = {"run_id": run_id, "final": evaluate(task, record["answer"]), "note": "No generated candidate"}
            else:
                evaluation = evaluate_run(task, record, RUN_DIR)
        except Exception:
            evaluation = {"run_id": run_id, "final": {"score": None, "passed": None}, "evaluation_error": traceback.format_exc()}
        (RUN_DIR / "evaluations" / f"{run_id}.json").write_text(json.dumps(evaluation, indent=1, ensure_ascii=False, default=str))
        totals, old = record["cost"], HISTORICAL.get(run_id, {})
        row = {"run_id": run_id, "task": task_id, "condition": condition,
               "score": evaluation["final"].get("score"), "passed": evaluation["final"].get("passed"),
               "seconds": round(totals["wall_ms"] / 1000, 1), "calls": totals["calls"], "tokens": totals["total_tokens"],
               "cycles": (end or {}).get("iterations"), "stop": (end or {}).get("termination"),
               "gate": (end or {}).get("quality_gate"), "withheld": bool(end) and not record["answer"],
               "error": bool(error),
               "old_score": float(old["external_score"]) if old.get("external_score") else None,
               "old_calls": int(old["total_model_calls"]) if old.get("total_model_calls") else None,
               "old_tokens": int(old["tokens"]) if old.get("tokens") else None}
        RESULTS.append(row)
        print({k: row[k] for k in ("score", "passed", "seconds", "calls", "tokens", "stop", "old_score")}, flush=True)
SUMMARY["results"] = RESULTS

## 7. Results

In [ ]:
def show_table(rows):
    cols = ("task", "condition", "score", "old_score", "passed", "seconds", "calls", "old_calls", "tokens", "stop", "withheld")
    width = {c: max(len(c), *(len(str(r.get(c))) for r in rows)) for c in cols}
    print(" | ".join(c.ljust(width[c]) for c in cols))
    for r in rows:
        print(" | ".join(str(r.get(c)).ljust(width[c]) for c in cols))
show_table(RESULTS)

print("\nTotals by condition (old = historical pilot, unfixed engine, same tasks and seeds):")
for condition in CONDITIONS:
    rows = [r for r in RESULTS if r["condition"] == condition]
    new = sum(r["score"] or 0 for r in rows)
    old = sum(r["old_score"] or 0 for r in rows if r["old_score"] is not None)
    print(f"  {condition:20s} score {new:.1f}/{len(rows)} (old {old:.1f}) | passed {sum(bool(r['passed']) for r in rows)}/{len(rows)}"
          f" | {sum(r['calls'] for r in rows)} calls | {sum(r['tokens'] or 0 for r in rows):,} tokens"
          f" | {sum(r['seconds'] for r in rows):.0f} s")
print("\nSame model build as the historical pilot:", SUMMARY["same_model_build_as_pilot"])
print("One trial per task: differences of one task are not significant.")

## 8. Save the results

In [ ]:
SUMMARY["finished_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
sh("nvidia-smi > " + str(RUN_DIR / "nvidia_smi_end.txt"), check=False)
(RUN_DIR / "summary.json").write_text(json.dumps(SUMMARY, indent=2, default=str))
zip_path = shutil.make_archive(str(WORK / f"end_colab_run_{RUN_ID}"), "zip", RUN_DIR)
print("Zip:", zip_path)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = pathlib.Path("/content/drive/MyDrive") / DRIVE_FOLDER
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_path, dest)
    print("Copied to Google Drive:", dest / pathlib.Path(zip_path).name)
else:
    try:
        from google.colab import files
        files.download(zip_path)
    except ImportError:
        print("Not running in Colab; zip left at", zip_path)